# Load the Data

To see how to use kaggle on git hub see [source](https://github.com/Kaggle/kagglehub/blob/main/README.md#download-dataset).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [3]:
# Load a DataFrame with a specific version of a CSV
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "amitabhajoy/bengaluru-house-price-data",
    "Bengaluru_House_Data.csv"
)

Using Colab cache for faster access to the 'bengaluru-house-price-data' dataset.


In [4]:
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [5]:
df.shape

(13320, 9)

# Data Cleaning

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


size should be a number, total_sqft should be a number.

In [7]:
df.groupby('area_type')['area_type'].agg('count')

,area_type
area_type,
Built-up Area,2418
Carpet Area,87
Plot Area,2025
Super built-up Area,8790


In [8]:
# Drop columns which would not help in machine learning
df_droped = df.drop(['area_type', 'availability', 'society', 'balcony'], axis = 1)
df_droped.shape

(13320, 5)

In [9]:
df_droped.head()

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


In [10]:
df_droped.isnull().sum()

,0
location,1
size,16
total_sqft,0
bath,73
price,0


In [11]:
# Since the data is big i'll just delete all the cols that are null.
df_nonna = df_droped.dropna()
df_nonna.isnull().sum()

,0
location,0
size,0
total_sqft,0
bath,0
price,0


In [12]:
df_droped.shape[0], df_nonna.shape[0]

(13320, 13246)

In [13]:
df_nonna['size'].unique()

array(['2 BHK', '4 Bedroom', '3 BHK', '4 BHK', '6 Bedroom', '3 Bedroom',
       '1 BHK', '1 RK', '1 Bedroom', '8 Bedroom', '2 Bedroom',
       '7 Bedroom', '5 BHK', '7 BHK', '6 BHK', '5 Bedroom', '11 BHK',
       '9 BHK', '9 Bedroom', '27 BHK', '10 Bedroom', '11 Bedroom',
       '10 BHK', '19 BHK', '16 BHK', '43 Bedroom', '14 BHK', '8 BHK',
       '12 Bedroom', '13 BHK', '18 Bedroom'], dtype=object)

 our regression model would not understand this...

In [14]:
df_nonna['bhk'] = df_nonna['size'].apply(lambda x: int(x.split(' ')[0]))

/tmp/ipykernel_34452/111542672.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nonna['bhk'] = df_nonna['size'].apply(lambda x: int(x.split(' ')[0]))


In [15]:
df_nonna = df_nonna.drop(['size'], axis = 1)
df_nonna.head()

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056,2.0,39.07,2
1,Chikka Tirupathi,2600,5.0,120.00,4
2,Uttarahalli,1440,2.0,62.00,3
3,Lingadheeranahalli,1521,3.0,95.00,3
4,Kothanur,1200,2.0,51.00,2


In [16]:
df_nonna['bhk'].unique()

array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18])

In [17]:
df_nonna[df_nonna['bhk'] > 20]

,location,total_sqft,bath,price,bhk
1718,2Electronic City Phase II,8000,27.0,230.0,27
4684,Munnekollal,2400,40.0,660.0,43


Looks like an error in data.

A 23 room house has 8k sq_ft area... but a 43 roomed house has only 2.4k.

In [18]:
# Let's see what our total_sqft looks like
df_nonna['total_sqft'].unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      dtype=object)

`1133- 1348` is not a number, so thats why it was an object datatype.

In [19]:
# Function to check and convert our total_sqft col
def is_float(x):
  try :
    float(x)
  except:
    return False
  return True

In [20]:
#  all columns that cannot be converted to float values
df_nonna[~df_nonna['total_sqft'].apply(is_float)].head(20)

,location,total_sqft,bath,price,bhk
30,Yelahanka,2100 - 2850,4.0,186.000,4
122,Hebbal,3067 - 8156,4.0,477.000,4
137,8th Phase JP Nagar,1042 - 1105,2.0,54.005,2
165,Sarjapur,1145 - 1340,2.0,43.490,2
188,KR Puram,1015 - 1540,2.0,56.800,2
410,Kengeri,34.46Sq. Meter,1.0,18.500,1
549,Hennur Road,1195 - 1440,2.0,63.770,2
648,Arekere,4125Perch,9.0,265.000,9
661,Yelahanka,1120 - 1145,2.0,48.130,2
672,Bettahalsoor,3090 - 5002,4.0,445.000,4


In [21]:
#  Function to convert values in range
def convert_sqft_To_num(x):
  tokens = x.split(' - ')
  if len(tokens) == 2:
    return (float(tokens[0]) + float(tokens[1])) / 2
  try :
    return float(x)
  except:
    return None

In [22]:
df_area = df_nonna.copy()

In [23]:
df_area['total_sqft'] = df_area['total_sqft'].apply(convert_sqft_To_num)
df_area.head(50)

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056.00,2.0,39.07,2
1,Chikka Tirupathi,2600.00,5.0,120.00,4
2,Uttarahalli,1440.00,2.0,62.00,3
3,Lingadheeranahalli,1521.00,3.0,95.00,3
4,Kothanur,1200.00,2.0,51.00,2
5,Whitefield,1170.00,2.0,38.00,2
6,Old Airport Road,2732.00,4.0,204.00,4
7,Rajaji Nagar,3300.00,4.0,600.00,4
8,Marathahalli,1310.00,3.0,63.25,3
9,Gandhi Bazar,1020.00,6.0,370.00,6


In [24]:
df_area.loc[410]

,410
location,Kengeri
total_sqft,NaN
bath,1.0
price,18.5
bhk,1


In [25]:
df_area.isnull().sum()

,0
location,0
total_sqft,46
bath,0
price,0
bhk,0


Still we have 46 rows with no area in them.

# Feature Engineering

In [26]:
df_price = df_area.copy()

In [27]:
# since data is in lakh multiply with 100000 and divide by total area
df_price['price_per_sqft'] = round((df_price['price'] * 1e5) / df_price['total_sqft'], 2)
df_price.head()

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.81
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.38
2,Uttarahalli,1440.0,2.0,62.00,3,4305.56
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.89
4,Kothanur,1200.0,2.0,51.00,2,4250.00


In [28]:
len(df_price['location'].unique())

1304

In [29]:
df_price.shape

(13246, 6)

There are about 1300 unique locations in our data.

Working with these data for machine learning will have to one-hot encode them and having only 13000 rows will have a problem called `The curse of dimensionality`.

So we will try to take only the names of the common locations and drop the rest in "other" location.

In [30]:
df_price.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13246 entries, 0 to 13319
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   location        13246 non-null  object 
 1   total_sqft      13200 non-null  float64
 2   bath            13246 non-null  float64
 3   price           13246 non-null  float64
 4   bhk             13246 non-null  int64  
 5   price_per_sqft  13200 non-null  float64
dtypes: float64(4), int64(1), object(1)
memory usage: 1.2+ MB


In [31]:
# Strip data of any spaces
df_price['location'] = df_price['location'].apply(lambda x : x.strip())
df_price.head()

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.81
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.38
2,Uttarahalli,1440.0,2.0,62.00,3,4305.56
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.89
4,Kothanur,1200.0,2.0,51.00,2,4250.00


In [32]:
location_stats = df_price.groupby('location')['location'].agg('count').sort_values(ascending = False)
location_stats

,location
location,
Whitefield,535
Sarjapur Road,392
Electronic City,304
Kanakpura Road,266
Thanisandra,236
...,...
poornaprajna layout,1
pavitra paradise,1
near Ramanashree California resort,1


In [33]:
len(location_stats[location_stats <= 10])

1052

In [34]:
location_stats_less_than_10 = location_stats[ location_stats <= 10]
location_stats_less_than_10

,location
location,
Kalkere,10
Sadashiva Nagar,10
BTM 1st Stage,10
Basapura,10
Gunjur Palya,10
...,...
poornaprajna layout,1
pavitra paradise,1
near Ramanashree California resort,1


In [35]:
df_price['location'] = df_price['location'].apply(lambda x: 'Other' if x in location_stats_less_than_10 else x)
len(df_price['location'].unique())

242

In [36]:
df_price.head(15)

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.81
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.38
2,Uttarahalli,1440.0,2.0,62.00,3,4305.56
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.89
4,Kothanur,1200.0,2.0,51.00,2,4250.00
5,Whitefield,1170.0,2.0,38.00,2,3247.86
6,Old Airport Road,2732.0,4.0,204.00,4,7467.06
7,Rajaji Nagar,3300.0,4.0,600.00,4,18181.82
8,Marathahalli,1310.0,3.0,63.25,3,4828.24
9,Other,1020.0,6.0,370.00,6,36274.51


# Outlier Removal

In [43]:
df_price

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.81
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.38
2,Uttarahalli,1440.0,2.0,62.00,3,4305.56
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.89
4,Kothanur,1200.0,2.0,51.00,2,4250.00
...,...,...,...,...,...,...
13315,Whitefield,3453.0,4.0,231.00,5,6689.83
13316,Other,3600.0,5.0,400.00,4,11111.11
13317,Raja Rajeshwari Nagar,1141.0,2.0,60.00,2,5258.55
13318,Padmanabhanagar,4689.0,4.0,488.00,4,10407.34


In [45]:
df1 = df_price[~((df_price.total_sqft / df_price.bhk) < 300)]
df1

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.81
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.38
2,Uttarahalli,1440.0,2.0,62.00,3,4305.56
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.89
4,Kothanur,1200.0,2.0,51.00,2,4250.00
...,...,...,...,...,...,...
13315,Whitefield,3453.0,4.0,231.00,5,6689.83
13316,Other,3600.0,5.0,400.00,4,11111.11
13317,Raja Rajeshwari Nagar,1141.0,2.0,60.00,2,5258.55
13318,Padmanabhanagar,4689.0,4.0,488.00,4,10407.34
